In [ ]:
import sys, os, glob, shutil, time, json, gc
import numpy as np, pandas as pd
t0 = time.perf_counter()
def log(m): print(f"[{time.perf_counter()-t0:6.0f}с] {m}", flush=True)
os.makedirs("/kaggle/working/src", exist_ok=True)
os.makedirs("/kaggle/working/models", exist_ok=True)
code = os.path.dirname(glob.glob("/kaggle/input/**/pair_features.py", recursive=True)[0])
for p in glob.glob(code + "/*.py"): shutil.copy(p, "/kaggle/working/src/")
for p in glob.glob(code + "/*.json"): shutil.copy(p, "/kaggle/working/models/")
# Версии из отправленного архива кладутся поверх: именно на них обучена структурная модель.
struct = os.path.dirname(glob.glob("/kaggle/input/**/pair_boost_hybrid.npz", recursive=True)[0])
for p in glob.glob(struct + "/*.py"): shutil.copy(p, "/kaggle/working/src/")
for p in glob.glob(struct + "/*.npz"): shutil.copy(p, "/kaggle/working/models/")
open("/kaggle/working/src/__init__.py", "a").close()
os.chdir("/kaggle/working"); sys.path.insert(0, "/kaggle/working")
from src.pair_features import build_matrix, feature_names
from src.measure_features import measures, compare_measures, MEASURE_FEATURES
from src.features import extract_model_features
from src.model import BoostedPairModel
from src.export_boost import export, save, predict_proba
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score
from scipy.stats import rankdata

fold = os.path.dirname(glob.glob("/kaggle/input/**/llm_valid_pairs.parquet", recursive=True)[0])
sc = os.path.dirname(glob.glob("/kaggle/input/**/ce_relaxed.npy", recursive=True)[0])
pairs = pd.read_parquet(fold + "/llm_valid_pairs.parquet")
items = pd.read_parquet(fold + "/llm_valid_items.parquet")
y = (pairs["target"].to_numpy() > 0).astype(np.int8)
cat = pairs["id1"].map(dict(zip(items.id, items.category.astype(str)))).astype(str).to_numpy()
known = sorted(set(items.category.astype(str)))
log(f"пар {len(pairs):,}, доля+ {y.mean():.3f}")

names = list(feature_names(False))
X = np.zeros((len(pairs), len(names)), dtype=np.float32)
for c in known:
    rows = np.flatnonzero(cat == c)
    if not len(rows): continue
    sub = items[items.category.astype(str) == c].reset_index(drop=True)
    X[rows] = build_matrix(sub, pairs.iloc[rows].reset_index(drop=True), known, with_neighbours=False)
    del sub; gc.collect()
log("парные признаки готовы")

t = time.perf_counter()
legacy = extract_model_features(pairs[["id1", "id2"]], items)
primary = BoostedPairModel("models/pair_boost_hybrid.npz").predict_probability(legacy, cat)
aux = BoostedPairModel("models/pair_boost_hybrid_aux.npz").predict_probability(legacy, cat)
structural = (0.8 * primary + 0.2 * aux).astype(np.float32)
del legacy; gc.collect()
log(f"структурная модель за {time.perf_counter()-t:.0f}с")

M = {int(i): measures(a) for i, a in zip(items.id, items.attributes)}
MX = np.array([[r[n] for n in MEASURE_FEATURES] for r in
               (compare_measures(M[x], M[z]) for x, z in zip(pairs.id1, pairs.id2))], dtype=np.float32)
CE = {n: np.load(f"{sc}/{n}.npy").astype(np.float32) for n in ("ce_relaxed", "ce_combo")}
codes = np.array([known.index(c) if c in known else -1 for c in cat], dtype=np.float32)

BASE_COLS = names + list(MEASURE_FEATURES) + ["ce_relaxed", "ce_combo", "category_code"]
BASE = np.column_stack([X, MX, CE["ce_relaxed"], CE["ce_combo"], codes]).astype(np.float64)
WITH_COLS = names + list(MEASURE_FEATURES) + ["ce_relaxed", "ce_combo", "structural", "category_code"]
WITH = np.column_stack([X, MX, CE["ce_relaxed"], CE["ce_combo"], structural, codes]).astype(np.float64)

masks = {c: cat == c for c in np.unique(cat)}
def rk(s):
    o = np.empty(len(s), np.float32)
    for m in masks.values(): o[m] = rankdata(s[m]) / m.sum()
    return o
def macro(s, rate=0.111, seeds=8):
    vals = []
    for seed in range(seeds):
        rng = np.random.default_rng(seed); per = []
        for m in masks.values():
            rows = np.flatnonzero(m); pos, neg = rows[y[rows]==1], rows[y[rows]==0]
            keep = min(len(pos), max(5, int(round(rate/(1-rate)*len(neg)))))
            ch = np.concatenate([rng.choice(pos, keep, replace=False), neg])
            per.append(average_precision_score(y[ch], s[ch]))
        vals.append(np.mean(per))
    return float(np.mean(vals)), float(np.std(vals))

FULL = np.column_stack([X, MX, CE["ce_relaxed"], CE["ce_combo"], structural, codes]).astype(np.float32)
log(f"матрица {FULL.shape}")
half = np.random.default_rng(5).permutation(len(y)) % 2

def crossfit(fit_predict):
    p = np.zeros(len(y))
    for h in (0, 1):
        tr, te = np.flatnonzero(half != h), np.flatnonzero(half == h)
        p[te] = fit_predict(FULL[tr], y[tr], FULL[te])
    return rk(p)

def sk(seed=0):
    def go(Xtr, ytr, Xte):
        m = HistGradientBoostingClassifier(max_iter=800, max_leaf_nodes=63, learning_rate=0.05,
                                           l2_regularization=1.0, early_stopping=False,
                                           random_state=seed).fit(Xtr, ytr)
        return m.predict_proba(Xte)[:, 1]
    return go

RESULTS = {}
t = time.perf_counter(); RESULTS["sklearn HistGB"] = macro(crossfit(sk()))
log(f"  sklearn HistGB          {RESULTS['sklearn HistGB'][0]:.6f} ± {RESULTS['sklearn HistGB'][1]:.6f}  ({time.perf_counter()-t:.0f}с)")

try:
    import lightgbm as lgb
    def go(Xtr, ytr, Xte):
        m = lgb.LGBMClassifier(n_estimators=1200, num_leaves=63, learning_rate=0.05,
                               colsample_bytree=0.8, subsample=0.8, subsample_freq=1,
                               reg_lambda=1.0, n_jobs=-1, verbose=-1, random_state=0).fit(Xtr, ytr)
        return m.predict_proba(Xte)[:, 1]
    t = time.perf_counter(); RESULTS["LightGBM"] = macro(crossfit(go))
    log(f"  LightGBM                {RESULTS['LightGBM'][0]:.6f} ± {RESULTS['LightGBM'][1]:.6f}  ({time.perf_counter()-t:.0f}с)")
except Exception as e:
    log(f"  LightGBM недоступен: {e}")

try:
    from catboost import CatBoostClassifier
    def go(Xtr, ytr, Xte):
        m = CatBoostClassifier(iterations=1500, depth=8, learning_rate=0.05, l2_leaf_reg=3.0,
                               verbose=0, thread_count=-1, random_seed=0).fit(Xtr, ytr)
        return m.predict_proba(Xte)[:, 1]
    t = time.perf_counter(); RESULTS["CatBoost"] = macro(crossfit(go))
    log(f"  CatBoost                {RESULTS['CatBoost'][0]:.6f} ± {RESULTS['CatBoost'][1]:.6f}  ({time.perf_counter()-t:.0f}с)")
except Exception as e:
    log(f"  CatBoost недоступен: {e}")

try:
    import xgboost as xgb
    def go(Xtr, ytr, Xte):
        m = xgb.XGBClassifier(n_estimators=1200, max_depth=8, learning_rate=0.05,
                              subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
                              tree_method="hist", n_jobs=-1, random_state=0).fit(Xtr, ytr)
        return m.predict_proba(Xte)[:, 1]
    t = time.perf_counter(); RESULTS["XGBoost"] = macro(crossfit(go))
    log(f"  XGBoost                 {RESULTS['XGBoost'][0]:.6f} ± {RESULTS['XGBoost'][1]:.6f}  ({time.perf_counter()-t:.0f}с)")
except Exception as e:
    log(f"  XGBoost недоступен: {e}")

# Усреднение нескольких sklearn с разными семенами: снижение дисперсии почти бесплатно,
# на инференсе это просто вдвое-втрое больше деревьев в том же формате выгрузки.
preds = [crossfit(sk(s)) for s in (0, 1, 2)]
RESULTS["sklearn, три семени"] = macro(rk(np.mean(preds, axis=0)))
log(f"  sklearn, три семени     {RESULTS['sklearn, три семени'][0]:.6f}")
log("\nитог: " + " | ".join(f"{k} {v[0]:.6f}" for k, v in sorted(RESULTS.items(), key=lambda kv: -kv[1][0])))
json.dump({k: v[0] for k, v in RESULTS.items()}, open("/kaggle/working/boosters.json", "w"), ensure_ascii=False, indent=1)
